# GPT-5.3-Codex with MCP and Web Search tools

This notebook demonstrates the use of Azure AI Foundry agent v2 with MCP and Web Search tools for the stock market investment Web site development.

### Step 1: Install Dependencies

!pip install azure-ai-projects azure-identity

### Step 2: Configuration

Update the Azure AI Foundry settings below with your project details.

In [1]:
# Import required packages
import os
import urllib.request
from urllib.error import URLError, HTTPError
from IPython.display import HTML, display
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    WebSearchTool,
    WebSearchApproximateLocation,
    MCPTool
)

In [2]:
# Set environment variables
PROJECT_ENDPOINT = os.getenv("AZURE_FOUNDRY_PROJECT_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("AZURE_FOUNDRY_CODEX_MODEL")
ALPHAVANTAGE_API_KEY = os.getenv("ALPHAVANTAGE_API_KEY")

MCP_SERVER_URL = f"https://mcp.alphavantage.co/mcp?apikey={ALPHAVANTAGE_API_KEY}"
AGENT_NAME = "stock-analyst-agent-v2"
SECTOR = "AI chips"

### Step 3: Create Foundry Agent v2 with Tools

In [3]:
# Initialise AIProjectClient
project_client = AIProjectClient(
    endpoint   = PROJECT_ENDPOINT,
    credential = DefaultAzureCredential()
)

openai_client = project_client.get_openai_client()
print("Foundry Project Client ready")

Foundry Project Client ready


In [4]:
# Create AI Agent with Web Search and MCP tools
agent = project_client.agents.create_version(
    agent_name  = AGENT_NAME,
    definition  = PromptAgentDefinition(
        model = DEPLOYMENT_NAME,
        instructions = """\
You are a financial analyst called The Alpha Finder.
You have two tools: web search and an alpha_finance MCP tool.

When given a sector:
1. Use web search to identify the top 5 publicly listed companies in that sector RIGHT NOW
   by market capitalisation. Extract their ticker symbols.
2. For each ticker, use the MCP tool to fetch live fundamentals and recent news.
3. Rank them by investment attractiveness with clear rationale.
4. Output ONLY a single self-contained dark-theme HTML dashboard showing the full analysis.
   No explanation, no markdown fences — just the HTML.
""",
        tools = [
            WebSearchTool(
                user_location = WebSearchApproximateLocation(
                    country = "GB",
                    city = "London"
                )
            ),
            MCPTool(
                server_label = "alpha_finance",
                server_url = MCP_SERVER_URL,
                require_approval = "never"
            ),
        ],
    ),
    description = "Web search + live MCP data + HTML generation",
)

print(f"Agent: {agent.name} v{agent.version}")

Agent: stock-analyst-agent-v2 v1


### Step 4: Test the Agent

In [5]:
# Run the agent to generate the stock dashboard
conversation = openai_client.conversations.create(
    items=[{
        "type": "message",
        "role": "user",
        "content": f"Analyse the {SECTOR} sector and generate the dashboard."
    }],
)
response = openai_client.responses.create(
    conversation = conversation.id,
    input = "",
    extra_body = {
        "agent_reference": {
            "name": agent.name,
            "type": "agent_reference"
        }
    },
)

html_output = response.output_text
print(f"Done — {len(html_output):,} chars | status: {response.status}")

Done — 10,777 chars | status: completed


In [6]:
# Save and display HTML output
output_file = f"alpha_finder_{SECTOR.lower().replace(' ', '_')}.html"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(html_output)
print(f"Saved → {output_file}")

display(HTML(html_output))

Saved → alpha_finder_ai_chips.html


### Step 5: Housekeeping

In [ ]:
# Delete agent and close clients
project_client.agents.delete_version(
    agent_name = agent.name,
    agent_version = agent.version)
print(f"Agent deleted: {agent.name}")

openai_client.close()
project_client.close()
print("All clients closed")